In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
from transformers import Dinov2Model, Dinov2Config
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from collections import Counter, defaultdict
import shutil
from pathlib import Path

DATA_DIR = "RS_images_2800"
for p in Path(DATA_DIR).rglob(".ipynb_checkpoints"):
    shutil.rmtree(p)

os.environ['http_proxy'] = 'http://127.0.0.1:7890'
os.environ['https_proxy'] = 'http://127.0.0.1:7890'

plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Liberation Sans"]
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# 1. 配置超参数与设备
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS = 15
LR = 1e-5
DATASET_NAME = "RSSCN7"
CLASSES = sorted(os.listdir(DATA_DIR))
NUM_CLASSES = len(CLASSES)

In [ ]:
# 2. 数据预处理
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
valid_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 3. 加载数据集
full_dataset = datasets.ImageFolder(root=DATA_DIR)
train_size = int(0.8 * len(full_dataset))
valid_size = len(full_dataset) - train_size
train_dataset_indices, valid_dataset_indices = random_split(
    range(len(full_dataset)), [train_size, valid_size]
)

class DatasetWithTransform(Dataset):
    def __init__(self, dataset, indices, transform):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __getitem__(self, idx):
        img, label = self.dataset[self.indices[idx]]
        if self.transform:
            img = self.transform(img)
        return img, label

    def __len__(self):
        return len(self.indices)

train_dataset = DatasetWithTransform(full_dataset, train_dataset_indices, train_transform)
valid_dataset = DatasetWithTransform(full_dataset, valid_dataset_indices, valid_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# 可视化展示函数
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
FIG_DIR = "RSSCN7_exp_fig"


def _safe_file_component(text):
    text = str(text)
    text = text.encode("ascii", "ignore").decode("ascii") or "figure"
    return "".join(ch if (ch.isalnum() or ch in "._-") else "_" for ch in text)


def _figure_filename(filename):
    prefix = globals().get("DATASET_NAME", "DINOv2_RS")
    prefix = _safe_file_component(prefix)
    filename = _safe_file_component(filename)
    if prefix and not filename.startswith(prefix + "_"):
        filename = f"{prefix}_{filename}"
    return filename


def save_and_show(fig, filename, dpi=220):
    """保存图像"""
    os.makedirs(FIG_DIR, exist_ok=True)
    path = os.path.join(FIG_DIR, _figure_filename(filename))
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved figure: {path}")
    return path


def get_display_class_names(class_names):
    display_names = []
    for i, name in enumerate(class_names):
        text = str(name).replace("_", " ")
        ascii_text = text.encode("ascii", "ignore").decode("ascii").strip()
        display_names.append(ascii_text if ascii_text else f"Class {i}")
    return display_names


def _to_index_list(indices):
    return [int(indices[i]) for i in range(len(indices))]


def _dataset_targets(raw_dataset):
    if hasattr(raw_dataset, "targets"):
        return [int(x) for x in raw_dataset.targets]
    return [int(raw_dataset[i][1]) for i in range(len(raw_dataset))]


def _targets_from_indices(raw_dataset, indices):
    targets = _dataset_targets(raw_dataset)
    return [targets[i] for i in _to_index_list(indices)]


def denormalize_tensor(img_tensor):
    """将经过 ImageNet Normalize 的 Tensor 转回可显示的 HWC RGB 图像。"""
    img = img_tensor.detach().cpu() * IMAGENET_STD + IMAGENET_MEAN
    img = img.clamp(0, 1).permute(1, 2, 0).numpy()
    return img


def plot_dataset_class_distribution(raw_dataset, train_dataset, valid_dataset, class_names):
    """展示全量、训练集和验证集的类别分布。"""
    display_names = get_display_class_names(class_names)
    full_targets = _dataset_targets(raw_dataset)
    train_targets = _targets_from_indices(raw_dataset, train_dataset.indices)
    valid_targets = _targets_from_indices(raw_dataset, valid_dataset.indices)

    def counts(targets):
        c = Counter(targets)
        return np.array([c.get(i, 0) for i in range(len(class_names))])

    full_counts = counts(full_targets)
    train_counts = counts(train_targets)
    valid_counts = counts(valid_targets)

    x = np.arange(len(class_names))
    width = 0.25
    fig, ax = plt.subplots(figsize=(max(10, len(class_names) * 0.65), 5))
    ax.bar(x - width, full_counts, width=width, label="Full")
    ax.bar(x, train_counts, width=width, label="Train")
    ax.bar(x + width, valid_counts, width=width, label="Validation")
    ax.set_xticks(x)
    ax.set_xticklabels(display_names, rotation=45, ha="right")
    ax.set_ylabel("Sample Count")
    ax.set_title("Dataset Class Distribution")
    ax.legend()
    fig.tight_layout()
    save_and_show(fig, "dataset_class_distribution.png")

    print(f"Full: {len(full_targets)} | Train: {len(train_targets)} | Validation: {len(valid_targets)}")


def show_dataset_samples(raw_dataset, class_names, samples_per_class=2, max_classes=None, seed=42):
    """按类别展示原始图像样本。"""
    display_names = get_display_class_names(class_names)
    rng = random.Random(seed)
    targets = _dataset_targets(raw_dataset)
    by_class = defaultdict(list)
    for idx, label in enumerate(targets):
        by_class[label].append(idx)

    selected_classes = list(range(len(class_names)))
    if max_classes is not None:
        selected_classes = selected_classes[:max_classes]

    rows = len(selected_classes)
    cols = samples_per_class
    fig, axes = plt.subplots(rows, cols, figsize=(max(4, cols * 3), max(3, rows * 2.2)))
    axes = np.array(axes).reshape(rows, cols)

    for row, class_idx in enumerate(selected_classes):
        candidates = by_class.get(class_idx, [])
        chosen = rng.sample(candidates, k=min(samples_per_class, len(candidates))) if candidates else []
        for col in range(cols):
            ax = axes[row, col]
            ax.axis("off")
            if col < len(chosen):
                img, _ = raw_dataset[chosen[col]]
                ax.imshow(img)
                title = display_names[class_idx] if col == 0 else ""
                ax.set_title(title, fontsize=9)
    fig.suptitle("Raw Samples by Class", y=1.002)
    fig.tight_layout()
    save_and_show(fig, "raw_samples_by_class.png")


def show_augmentation_examples(raw_dataset, transform, class_names, sample_index=0, num_augmented=5):
    """展示同一张图像经过训练增强后的多个视图。"""
    display_names = get_display_class_names(class_names)
    img, label = raw_dataset[sample_index]
    total = num_augmented + 1
    fig, axes = plt.subplots(1, total, figsize=(max(6, total * 2.6), 3))
    axes = np.array(axes).reshape(1, total)[0]

    axes[0].imshow(img)
    axes[0].set_title(f"Original\n{display_names[label]}", fontsize=9)
    axes[0].axis("off")

    for i in range(1, total):
        aug = transform(img)
        axes[i].imshow(denormalize_tensor(aug))
        axes[i].set_title(f"Augmented {i}", fontsize=9)
        axes[i].axis("off")
    fig.suptitle("Data Augmentation Examples")
    fig.tight_layout()
    save_and_show(fig, "augmentation_examples.png")


def plot_training_history(history):
    """展示训练损失、验证损失、验证准确率和学习率曲线。"""
    if not history or len(history.get("epoch", [])) == 0:
        print("History is empty. Run the training loop first.")
        return

    epochs = history["epoch"]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(epochs, history["train_loss"], marker="o", label="Train Loss")
    if "val_loss" in history:
        ax.plot(epochs, history["val_loss"], marker="o", label="Validation Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Training and Validation Loss")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    save_and_show(fig, "training_validation_loss.png")

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(epochs, history["val_acc"], marker="o", label="Validation Accuracy")
    best_epoch = int(np.argmax(history["val_acc"]))
    ax.scatter([epochs[best_epoch]], [history["val_acc"][best_epoch]], s=80, label="Best")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_title("Validation Accuracy")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    save_and_show(fig, "validation_accuracy.png")

    if "lr" in history:
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.step(epochs, history["lr"], where="mid", label="Learning Rate")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Learning Rate")
        ax.set_title("Learning Rate Schedule")
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        save_and_show(fig, "learning_rate_schedule.png")


def evaluate_model_for_visualization(model, data_loader, class_names, device, criterion=None, max_store_images=160):
    """展示验证集预测、置信度、样本示例等"""
    model.eval()
    y_true, y_pred, confidences = [], [], []
    all_probs = []
    sample_images, sample_true, sample_pred, sample_conf = [], [], [], []
    stored = 0
    total_loss = 0.0
    total_count = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            if criterion is not None:
                loss = criterion(outputs, labels)
                total_loss += loss.item() * images.size(0)
            probs = torch.softmax(outputs, dim=1)
            conf, preds = torch.max(probs, dim=1)

            y_true.extend(labels.detach().cpu().numpy().tolist())
            y_pred.extend(preds.detach().cpu().numpy().tolist())
            confidences.extend(conf.detach().cpu().numpy().tolist())
            all_probs.append(probs.detach().cpu())
            total_count += images.size(0)

            if stored < max_store_images:
                take = min(max_store_images - stored, images.size(0))
                sample_images.append(images[:take].detach().cpu())
                sample_true.extend(labels[:take].detach().cpu().numpy().tolist())
                sample_pred.extend(preds[:take].detach().cpu().numpy().tolist())
                sample_conf.extend(conf[:take].detach().cpu().numpy().tolist())
                stored += take


    y_true = np.array(y_true, dtype=int)
    y_pred = np.array(y_pred, dtype=int)
    confidences = np.array(confidences, dtype=float)
    all_probs = torch.cat(all_probs, dim=0).numpy() if all_probs else np.empty((0, len(class_names)))
    sample_images = torch.cat(sample_images, dim=0) if sample_images else torch.empty(0)

    eval_loss = total_loss / total_count if (criterion is not None and total_count > 0) else None
    eval_acc = float((y_true == y_pred).mean()) if len(y_true) > 0 else 0.0
    return {
        "y_true": y_true,
        "y_pred": y_pred,
        "confidences": confidences,
        "probs": all_probs,
        "loss": eval_loss,
        "acc": eval_acc,
        "sample_images": sample_images,
        "sample_true": np.array(sample_true, dtype=int),
        "sample_pred": np.array(sample_pred, dtype=int),
        "sample_conf": np.array(sample_conf, dtype=float),
    }


def compute_confusion_matrix(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        if 0 <= int(t) < num_classes and 0 <= int(p) < num_classes:
            cm[int(t), int(p)] += 1
    return cm


def plot_confusion_matrix(y_true, y_pred, class_names, normalize=True):
    """展示混淆矩阵。"""
    display_names = get_display_class_names(class_names)
    cm = compute_confusion_matrix(y_true, y_pred, len(class_names))
    if normalize:
        denom = cm.sum(axis=1, keepdims=True)
        cm_show = np.divide(cm, denom, out=np.zeros_like(cm, dtype=float), where=denom != 0)
        title = "Normalized Confusion Matrix"
        filename = "confusion_matrix_normalized.png"
    else:
        cm_show = cm
        title = "Confusion Matrix (Counts)"
        filename = "confusion_matrix_counts.png"

    fig, ax = plt.subplots(figsize=(max(7, len(class_names) * 0.55), max(6, len(class_names) * 0.5)))
    im = ax.imshow(cm_show, aspect="auto")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(np.arange(len(class_names)))
    ax.set_xticklabels(display_names, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_yticklabels(display_names)
    ax.set_xlabel("Predicted Class")
    ax.set_ylabel("True Class")
    ax.set_title(title)

    if len(class_names) <= 12:
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                text = f"{cm_show[i, j]:.2f}" if normalize else str(cm[i, j])
                ax.text(j, i, text, ha="center", va="center", fontsize=8)

    fig.tight_layout()
    save_and_show(fig, filename)
    return cm


def plot_per_class_accuracy(y_true, y_pred, class_names):
    """展示每个类别的 Top-1 准确率。"""
    display_names = get_display_class_names(class_names)
    cm = compute_confusion_matrix(y_true, y_pred, len(class_names))
    support = cm.sum(axis=1)
    acc = np.divide(np.diag(cm), support, out=np.zeros(len(class_names), dtype=float), where=support != 0)

    fig, ax = plt.subplots(figsize=(max(10, len(class_names) * 0.65), 4.8))
    ax.bar(np.arange(len(class_names)), acc)
    ax.set_xticks(np.arange(len(class_names)))
    ax.set_xticklabels(display_names, rotation=45, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Accuracy")
    ax.set_title("Per-Class Validation Accuracy")
    fig.tight_layout()
    save_and_show(fig, "per_class_validation_accuracy.png")

    order = np.argsort(acc)
    print("Lowest-accuracy classes:")
    for idx in order[: min(5, len(order))]:
        print(f"  {display_names[idx]}: acc={acc[idx]:.4f}, support={support[idx]}")
    return acc


def print_classification_report(y_true, y_pred, class_names):
    """打印Precision/Recall/F1/Support分类报告。"""
    display_names = get_display_class_names(class_names)
    cm = compute_confusion_matrix(y_true, y_pred, len(class_names))
    rows = []
    print(f"{'Class':<25}{'Precision':>12}{'Recall':>12}{'F1':>12}{'Support':>12}")
    print("-" * 73)
    for i, name in enumerate(display_names):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        support = cm[i, :].sum()
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        rows.append({"class": name, "precision": precision, "recall": recall, "f1": f1, "support": int(support)})
        print(f"{name:<25}{precision:>12.4f}{recall:>12.4f}{f1:>12.4f}{support:>12d}")
    macro_precision = float(np.mean([r["precision"] for r in rows])) if rows else 0.0
    macro_recall = float(np.mean([r["recall"] for r in rows])) if rows else 0.0
    macro_f1 = float(np.mean([r["f1"] for r in rows])) if rows else 0.0
    print("-" * 73)
    print(f"{'Macro Avg':<25}{macro_precision:>12.4f}{macro_recall:>12.4f}{macro_f1:>12.4f}{len(y_true):>12d}")
    return rows


def plot_confidence_distribution(y_true, y_pred, confidences):
    """展示正确/错误预测的最大softmax置信度分布"""
    if len(confidences) == 0:
        print("No confidence scores are available.")
        return
    correct = np.asarray(y_true) == np.asarray(y_pred)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    bins = np.linspace(0, 1, 21)
    if correct.any():
        ax.hist(confidences[correct], bins=bins, alpha=0.65, label="Correct")
    if (~correct).any():
        ax.hist(confidences[~correct], bins=bins, alpha=0.65, label="Incorrect")
    ax.set_xlabel("Max Softmax Confidence")
    ax.set_ylabel("Sample Count")
    ax.set_title("Prediction Confidence Distribution")
    ax.legend()
    fig.tight_layout()
    save_and_show(fig, "prediction_confidence_distribution.png")


def plot_prediction_examples(eval_result, class_names, mode="errors", max_images=12):
    """展示验证集中预测正确或错误的样本。"""
    display_names = get_display_class_names(class_names)
    images = eval_result["sample_images"]
    y_true = eval_result["sample_true"]
    y_pred = eval_result["sample_pred"]
    conf = eval_result["sample_conf"]
    if images.numel() == 0:
        print("No cached images are available. Increase max_store_images in evaluate_model_for_visualization.")
        return

    if mode == "errors":
        idxs = np.where(y_true != y_pred)[0]
        title = "Incorrect Prediction Examples"
        filename = "incorrect_prediction_examples.png"
    elif mode == "correct":
        idxs = np.where(y_true == y_pred)[0]
        title = "Correct Prediction Examples"
        filename = "correct_prediction_examples.png"
    else:
        idxs = np.arange(len(y_true))
        title = "Prediction Examples"
        filename = "prediction_examples.png"

    if len(idxs) == 0:
        print(f"No samples are available for: {title}.")
        return

    idxs = idxs[:max_images]
    cols = min(4, len(idxs))
    rows = math.ceil(len(idxs) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 3.2))
    axes = np.array(axes).reshape(rows, cols)
    for ax in axes.ravel():
        ax.axis("off")

    for ax, idx in zip(axes.ravel(), idxs):
        ax.imshow(denormalize_tensor(images[idx]))
        true_name = display_names[int(y_true[idx])]
        pred_name = display_names[int(y_pred[idx])]
        ax.set_title(f"True: {true_name}\nPred: {pred_name}\nConf: {conf[idx]:.2f}", fontsize=8)
        ax.axis("off")
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    save_and_show(fig, filename)






def _safe_hidden_state_index(hidden_states, index):
    return max(0, min(int(index), len(hidden_states) - 1))


def _normalize_map(array):
    array = np.asarray(array, dtype=np.float32)
    array = array - float(array.min())
    max_value = float(array.max())
    if max_value > 1e-12:
        array = array / max_value
    return array


def extract_token_feature_map(model, image_tensor, device, layer_index=12):
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            image_batch = image_tensor.unsqueeze(0).to(device)
            outputs = model.backbone(pixel_values=image_batch, output_hidden_states=True)
            hidden_states = outputs.hidden_states
            layer_index = _safe_hidden_state_index(hidden_states, layer_index)
            hidden = hidden_states[layer_index][0]
            patch_tokens = hidden[1:, :]
            token_values = patch_tokens.abs().mean(dim=-1).detach().cpu().float()
            grid_size = int(math.sqrt(token_values.numel()))
            if grid_size < 1:
                raise ValueError("No patch tokens were found for feature-map visualization.")
            token_values = token_values[: grid_size * grid_size]
            feature_map = token_values.view(grid_size, grid_size).numpy()
            return _normalize_map(feature_map)
    finally:
        if was_training:
            model.train()


def record_feature_map_snapshot(model, image_tensor, label, device, history, epoch, layer_index=12):
    """记录特征图"""
    feature_map = extract_token_feature_map(model, image_tensor, device, layer_index=layer_index)
    history.append({
        "epoch": int(epoch),
        "feature_map": feature_map,
        "label": int(label),
        "layer_index": int(layer_index),
    })


def plot_feature_map_evolution(feature_map_history, image_tensor, label, class_names, max_snapshots=6):
    """绘制一个样本在整个训练过程中的token激活图."""

    display_names = get_display_class_names(class_names)
    if len(feature_map_history) > max_snapshots:
        selected_indices = np.unique(
            np.linspace(0, len(feature_map_history) - 1, num=max_snapshots, dtype=int)
        )
        snapshots = [feature_map_history[i] for i in selected_indices]
    else:
        snapshots = list(feature_map_history)

    num_maps = len(snapshots)
    total_axes = num_maps + 1  # input image + feature maps

    fig_width = max(10.0, 2.35 * total_axes + 0.9)
    fig = plt.figure(figsize=(fig_width, 3.7), constrained_layout=False)
    gs = fig.add_gridspec(
        nrows=1,
        ncols=total_axes + 1,
        width_ratios=[1.0] * total_axes + [0.04],
        wspace=0.08,
    )

    axes = [fig.add_subplot(gs[0, i]) for i in range(total_axes)]
    cax = fig.add_subplot(gs[0, total_axes])

    axes[0].imshow(denormalize_tensor(image_tensor))
    axes[0].set_title(f"Input Image\nTrue: {display_names[int(label)]}", fontsize=9, pad=7)
    axes[0].axis("off")

    last_im = None
    for ax, snapshot in zip(axes[1:], snapshots):
        last_im = ax.imshow(snapshot["feature_map"], aspect="auto", vmin=0.0, vmax=1.0)
        epoch = int(snapshot.get("epoch", -1))
        title = "Before Training" if epoch == 0 else f"Epoch {epoch}"
        ax.set_title(title, fontsize=9, pad=7)
        ax.axis("off")

    if last_im is not None:
        cbar = fig.colorbar(last_im, cax=cax)
        cbar.ax.set_ylabel("Normalized Activation", rotation=270, labelpad=22)
        cbar.ax.tick_params(labelsize=8)
    else:
        cax.axis("off")

    fig.suptitle("Feature Map Evolution for One Sample", fontsize=12, y=0.98)
    fig.subplots_adjust(
        left=0.018,
        right=0.982,
        top=0.78,
        bottom=0.08,
        wspace=0.08,
    )
    save_and_show(fig, "feature_map_evolution.png")


def _best_grid_from_token_count(num_tokens):
    num_tokens = int(num_tokens)
    grid_size = int(math.sqrt(num_tokens))
    if grid_size * grid_size == num_tokens:
        return grid_size, grid_size, num_tokens

    # Fallback: use the largest square that can be built from the available tokens.
    grid_size = max(1, int(math.sqrt(max(1, num_tokens))))
    return grid_size, grid_size, grid_size * grid_size


def _patch_tokens_from_hidden_state(hidden_state):
    if hidden_state.dim() == 3:
        tokens = hidden_state[0]
    else:
        tokens = hidden_state

    if tokens.dim() != 2:
        raise ValueError(f"Expected a token tensor with shape [num_tokens, hidden_dim], got {tuple(tokens.shape)}.")

    num_tokens = int(tokens.shape[0])
    if num_tokens > 1:
        grid_h, grid_w, used_tokens = _best_grid_from_token_count(num_tokens - 1)
        if used_tokens == num_tokens - 1:
            return tokens[1: 1 + used_tokens, :], (grid_h, grid_w)

    grid_h, grid_w, used_tokens = _best_grid_from_token_count(num_tokens)
    return tokens[:used_tokens, :], (grid_h, grid_w)


def _activation_map_from_tokens_or_scores(values, grid_hw=None, reducer="mean_abs"):
    if not torch.is_tensor(values):
        values = torch.tensor(values)

    if values.dim() == 3:
        values = values[0]

    if values.dim() == 2:
        if reducer == "l2":
            patch_values = torch.linalg.norm(values.detach().float(), dim=-1)
        else:
            patch_values = values.detach().float().abs().mean(dim=-1)
    else:
        patch_values = values.detach().float().reshape(-1)

    if grid_hw is None:
        grid_h, grid_w, used_tokens = _best_grid_from_token_count(int(patch_values.numel()))
    else:
        grid_h, grid_w = int(grid_hw[0]), int(grid_hw[1])
        used_tokens = grid_h * grid_w

    patch_values = patch_values[:used_tokens]
    feature_map = patch_values.view(grid_h, grid_w).cpu().numpy()
    return _normalize_map(feature_map)


def extract_key_position_feature_maps(
    model,
    image_tensor,
    class_names,
    device,
    target_class=None,
    selected_layer_numbers=(3, 6, 9, 12),
):
    """提取DINOv2遥感分类器关键位置的特征图。"""
    display_names = get_display_class_names(class_names)
    was_training = model.training
    model.eval()

    try:
        with torch.no_grad():
            image_batch = image_tensor.unsqueeze(0).to(device)
            outputs = model.backbone(pixel_values=image_batch, output_hidden_states=True)
            hidden_states = outputs.hidden_states

            stage_maps = []

            # 1) Patch embedding / input token representation.
            embed_index = _safe_hidden_state_index(hidden_states, 0)
            embed_patch_tokens, grid_hw = _patch_tokens_from_hidden_state(hidden_states[embed_index])
            stage_maps.append({
                "title": "Patch Embedding",
                "feature_map": _activation_map_from_tokens_or_scores(embed_patch_tokens, grid_hw=grid_hw),
            })

            # 2) DINOv2 layers selected by the original forward function.
            # Original logic: [all_hidden_states[i-1] for i in [3, 6, 9, 12]].
            selected_patch_tokens = []
            selected_cls_tokens = []

            for layer_number in selected_layer_numbers:
                hidden_index = _safe_hidden_state_index(hidden_states, int(layer_number) - 1)
                patch_tokens, grid_hw = _patch_tokens_from_hidden_state(hidden_states[hidden_index])
                cls_token = hidden_states[hidden_index][:, 0, :]

                selected_patch_tokens.append(patch_tokens.unsqueeze(0))
                selected_cls_tokens.append(cls_token)

                stage_maps.append({
                    "title": f"DINO Layer {int(layer_number)}",
                    "feature_map": _activation_map_from_tokens_or_scores(patch_tokens, grid_hw=grid_hw),
                })

            if not selected_patch_tokens:
                print("No selected DINOv2 layers were available for key-position visualization.")
                return None

            # 3) Learnable multi-scale fusion, visualized on patch tokens as a spatial proxy.
            if hasattr(model, "layer_weights") and int(model.layer_weights.numel()) == len(selected_patch_tokens):
                layer_norm_weights = torch.softmax(model.layer_weights, dim=0)
            else:
                layer_norm_weights = torch.ones(len(selected_patch_tokens), device=device) / len(selected_patch_tokens)

            fused_patch_tokens = torch.zeros_like(selected_patch_tokens[0])
            fused_cls_token = torch.zeros_like(selected_cls_tokens[0])

            for idx, (patch_tokens, cls_token) in enumerate(zip(selected_patch_tokens, selected_cls_tokens)):
                fused_patch_tokens = fused_patch_tokens + layer_norm_weights[idx] * patch_tokens
                fused_cls_token = fused_cls_token + layer_norm_weights[idx] * cls_token

            stage_maps.append({
                "title": "Multi-Scale Fusion",
                "feature_map": _activation_map_from_tokens_or_scores(fused_patch_tokens, grid_hw=grid_hw),
            })

            # 4) Adapter output and adapter-fused representation.
            if hasattr(model, "adapter"):
                adapter_patch_tokens = model.adapter(fused_patch_tokens)
                adapter_cls_token = model.adapter(fused_cls_token)

                stage_maps.append({
                    "title": "Adapter Output",
                    "feature_map": _activation_map_from_tokens_or_scores(adapter_patch_tokens, grid_hw=grid_hw),
                })
            else:
                adapter_patch_tokens = fused_patch_tokens
                adapter_cls_token = fused_cls_token

            if hasattr(model, "adapt_weights") and int(model.adapt_weights.numel()) >= 2:
                adapt_norm_weights = torch.softmax(model.adapt_weights, dim=0)
                adapted_patch_tokens = adapt_norm_weights[0] * adapter_patch_tokens + adapt_norm_weights[1] * fused_patch_tokens
                adapted_cls_token = adapt_norm_weights[0] * adapter_cls_token + adapt_norm_weights[1] * fused_cls_token
            else:
                adapted_patch_tokens = adapter_patch_tokens
                adapted_cls_token = adapter_cls_token

            stage_maps.append({
                "title": "Adapter-Fused Tokens",
                "feature_map": _activation_map_from_tokens_or_scores(adapted_patch_tokens, grid_hw=grid_hw),
            })

            # 5) Classifier and memory-enhanced class evidence.
            cls_logits = model.classifier(adapted_cls_token)

            memory_available = (
                hasattr(model, "memory_bank")
                and hasattr(model, "memory_weight")
                and model.memory_bank.detach().abs().sum().item() > 1e-8
            )

            if memory_available:
                cls_memory_logits = adapted_cls_token @ model.memory_bank.detach().clone().T
                cls_logits = cls_logits + model.memory_weight * cls_memory_logits

            cls_probs = torch.softmax(cls_logits, dim=1)
            pred_idx = int(torch.argmax(cls_probs, dim=1).item())
            confidence = float(cls_probs[0, pred_idx].detach().cpu())

            target_idx = pred_idx if target_class is None else int(target_class)
            target_idx = max(0, min(target_idx, cls_logits.shape[1] - 1))

            patch_logits = model.classifier(adapted_patch_tokens)
            classifier_response = patch_logits[0, :, target_idx]

            stage_maps.append({
                "title": f"Classifier Response\nTarget: {display_names[target_idx]}",
                "feature_map": _activation_map_from_tokens_or_scores(classifier_response, grid_hw=grid_hw),
            })

            if memory_available:
                patch_memory_logits = adapted_patch_tokens @ model.memory_bank.detach().clone().T
                memory_response = patch_memory_logits[0, :, target_idx]
                final_response = patch_logits[0, :, target_idx] + model.memory_weight.detach() * memory_response

                stage_maps.append({
                    "title": "Memory Similarity",
                    "feature_map": _activation_map_from_tokens_or_scores(memory_response, grid_hw=grid_hw),
                })

                stage_maps.append({
                    "title": "Final Class Evidence",
                    "feature_map": _activation_map_from_tokens_or_scores(final_response, grid_hw=grid_hw),
                })
            else:
                print("Memory-stage maps are skipped because the memory bank is empty or unavailable.")

            return {
                "stage_maps": stage_maps,
                "pred_idx": pred_idx,
                "target_idx": target_idx,
                "confidence": confidence,
            }

    finally:
        if was_training:
            model.train()


def plot_key_position_feature_maps(
    model,
    dataset,
    class_names,
    device,
    sample_index=0,
    target_class=None,
    selected_layer_numbers=(3, 6, 9, 12),
    max_columns=5,
):
    """绘制网络关键位置的特征图。"""
    if len(dataset) == 0:
        print("The dataset is empty. Key-position feature-map visualization is skipped.")
        return

    display_names = get_display_class_names(class_names)
    sample_index = max(0, min(int(sample_index), len(dataset) - 1))
    image_tensor, true_label = dataset[sample_index]

    result = extract_key_position_feature_maps(
        model=model,
        image_tensor=image_tensor,
        class_names=class_names,
        device=device,
        target_class=target_class,
        selected_layer_numbers=selected_layer_numbers,
    )

    if result is None:
        return

    stage_maps = result["stage_maps"]
    pred_idx = int(result["pred_idx"])
    target_idx = int(result["target_idx"])

    pred_name = display_names[pred_idx]
    target_name = display_names[target_idx]
    true_name = display_names[int(true_label)]

    panels = [{
        "title": f"Input Image\nTrue: {true_name}",
        "image": denormalize_tensor(image_tensor),
        "is_input": True,
    }]

    panels.extend({
        "title": item["title"],
        "image": item["feature_map"],
        "is_input": False,
    } for item in stage_maps)

    num_panels = len(panels)
    ncols = max(1, min(int(max_columns), num_panels))
    nrows = int(math.ceil(num_panels / ncols))

    # Dedicated colorbar column prevents overlap with the rightmost subplot.
    fig_width = max(12.0, 2.65 * ncols + 0.9)
    fig_height = max(5.0, 2.85 * nrows + 0.4)

    fig = plt.figure(figsize=(fig_width, fig_height), constrained_layout=False)

    gs = fig.add_gridspec(
        nrows=nrows,
        ncols=ncols + 1,
        width_ratios=[1.0] * ncols + [0.055],
        wspace=0.12,
        hspace=0.40,
    )

    axes = []
    for row in range(nrows):
        for col in range(ncols):
            axes.append(fig.add_subplot(gs[row, col]))

    cax = fig.add_subplot(gs[:, ncols])

    last_im = None

    for panel_index, ax in enumerate(axes):
        if panel_index >= num_panels:
            ax.axis("off")
            continue

        panel = panels[panel_index]

        if panel["is_input"]:
            ax.imshow(panel["image"])
        else:
            last_im = ax.imshow(panel["image"], aspect="auto", vmin=0.0, vmax=1.0)

        ax.set_title(panel["title"], fontsize=8.8, pad=7)
        ax.axis("off")

    if last_im is not None:
        cbar = fig.colorbar(last_im, cax=cax)
        cbar.ax.set_ylabel("Normalized Activation", rotation=270, labelpad=24)
        cbar.ax.tick_params(labelsize=8)
    else:
        cax.axis("off")

    fig.suptitle(
        f"Key Network Position Feature Maps | Target: {target_name} | Pred: {pred_name} ({result['confidence']:.2f})",
        fontsize=12,
        y=0.98,
    )

    fig.subplots_adjust(
        left=0.025,
        right=0.965,
        top=0.88,
        bottom=0.06,
        wspace=0.12,
        hspace=0.40,
    )

    save_and_show(fig, "key_position_feature_maps.png")


def _logits_from_hidden_states_for_visualization(model, hidden_states):
    layer_indices = [2, 5, 8, 11]  # Mirrors [i-1 for i in [3, 6, 9, 12]] in model.forward.
    layers = [hidden_states[_safe_hidden_state_index(hidden_states, idx)][:, 0, :] for idx in layer_indices]

    if hasattr(model, "layer_weights"):
        layer_norm_weights = torch.softmax(model.layer_weights, dim=0)
    else:
        layer_norm_weights = torch.ones(len(layers), device=layers[0].device) / len(layers)

    weighted_feat = torch.zeros_like(layers[0])
    for i, feat in enumerate(layers):
        weighted_feat = weighted_feat + layer_norm_weights[i] * feat

    if hasattr(model, "adapter") and hasattr(model, "adapt_weights"):
        adapt_norm_weights = torch.softmax(model.adapt_weights, dim=0)
        adapted_feat = adapt_norm_weights[0] * model.adapter(weighted_feat) + adapt_norm_weights[1] * weighted_feat
    elif hasattr(model, "adapter"):
        adapted_feat = model.adapter(weighted_feat)
    else:
        adapted_feat = weighted_feat

    orig_logits = model.classifier(adapted_feat)
    if hasattr(model, "memory_bank") and hasattr(model, "memory_weight"):
        memory_bank_current = model.memory_bank.detach().clone()
        memory_logits = adapted_feat @ memory_bank_current.T
        logits = orig_logits + model.memory_weight * memory_logits
    else:
        logits = orig_logits
    return logits


def _compute_grad_cam_once(model, image_tensor, device, target_class=None, layer_index=10):
    image_batch = image_tensor.unsqueeze(0).to(device)
    model.zero_grad(set_to_none=True)

    outputs = model.backbone(pixel_values=image_batch, output_hidden_states=True)
    hidden_states = outputs.hidden_states
    layer_index = _safe_hidden_state_index(hidden_states, layer_index)
    target_tokens = hidden_states[layer_index]
    target_tokens.retain_grad()

    logits = _logits_from_hidden_states_for_visualization(model, hidden_states)
    probs = torch.softmax(logits, dim=1)
    pred_idx = int(torch.argmax(probs, dim=1).item())
    confidence = float(probs[0, pred_idx].detach().cpu())
    used_target = pred_idx if target_class is None else int(target_class)
    used_target = max(0, min(used_target, logits.shape[1] - 1))

    score = logits[0, used_target]
    score.backward()

    grads = target_tokens.grad
    if grads is None:
        return None

    activations = target_tokens.detach()
    grads_patch = grads[0, 1:, :]
    activations_patch = activations[0, 1:, :]
    if grads_patch.numel() == 0:
        return None

    weights = grads_patch.mean(dim=0)
    cam = (activations_patch * weights).sum(dim=-1)
    cam = torch.relu(cam)

    grid_size = int(math.sqrt(cam.numel()))
    if grid_size < 1:
        return None
    cam = cam[: grid_size * grid_size].view(grid_size, grid_size).detach().cpu().numpy()
    cam = _normalize_map(cam)
    return {
        "cam": cam,
        "pred_idx": pred_idx,
        "target_idx": used_target,
        "confidence": confidence,
        "layer_index": layer_index,
    }


def compute_grad_cam(model, image_tensor, device, target_class=None, preferred_layer_index=10):
    """计算Grad-CAM热力图"""
    was_training = model.training
    model.eval()
    best_result = None
    candidate_indices = [preferred_layer_index, preferred_layer_index - 1, preferred_layer_index - 2, 8, 5, 2, 1]
    seen = set()
    try:
        for candidate in candidate_indices:
            if candidate in seen or candidate < 0:
                continue
            seen.add(candidate)
            result = _compute_grad_cam_once(
                model=model,
                image_tensor=image_tensor,
                device=device,
                target_class=target_class,
                layer_index=candidate,
            )
            if result is None:
                continue
            best_result = result
            if float(np.max(result["cam"])) > 1e-6:
                break
        return best_result
    finally:
        model.zero_grad(set_to_none=True)
        if was_training:
            model.train()


def _upsample_map(array, size_hw):
    array_tensor = torch.tensor(array, dtype=torch.float32).view(1, 1, array.shape[0], array.shape[1])
    upsampled = F.interpolate(array_tensor, size=size_hw, mode="bilinear", align_corners=False)
    return upsampled.squeeze(0).squeeze(0).numpy()


def plot_grad_cam(model, dataset, class_names, device, sample_index=0, target_class=None, preferred_layer_index=10):
    """展示Grad-CAM热力图"""
    if len(dataset) == 0:
        print("The dataset is empty. Grad-CAM visualization is skipped.")
        return

    display_names = get_display_class_names(class_names)
    sample_index = max(0, min(int(sample_index), len(dataset) - 1))
    image_tensor, true_label = dataset[sample_index]

    result = compute_grad_cam(
        model=model,
        image_tensor=image_tensor,
        device=device,
        target_class=target_class,
        preferred_layer_index=preferred_layer_index,
    )

    if result is None:
        print("Grad-CAM could not be computed for this sample.")
        return

    cam = result["cam"]
    if float(np.max(cam)) <= 1e-6:
        print("The Grad-CAM map is close to zero. The selected layer may have weak gradient flow for this sample.")

    image_np = denormalize_tensor(image_tensor)
    cam_up = _upsample_map(cam, image_np.shape[:2])
    cam_up = np.clip(cam_up, 0.0, 1.0)

    pred_idx = int(result["pred_idx"])
    target_idx = int(result["target_idx"])

    pred_name = display_names[pred_idx]
    target_name = display_names[target_idx]
    true_name = display_names[int(true_label)]

    fig = plt.figure(figsize=(16, 4.8), constrained_layout=False)
    gs = fig.add_gridspec(
        nrows=1,
        ncols=4,
        width_ratios=[1.0, 1.0, 1.0, 0.045],
        wspace=0.16,
    )

    axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
    cax = fig.add_subplot(gs[0, 3])

    axes[0].imshow(image_np)
    axes[0].set_title(f"Input Image\nTrue: {true_name}", fontsize=10, pad=8)
    axes[0].axis("off")

    heatmap = axes[1].imshow(cam_up, aspect="auto", vmin=0.0, vmax=1.0)
    axes[1].set_title(f"Grad-CAM Heatmap\nTarget: {target_name}", fontsize=10, pad=8)
    axes[1].axis("off")

    axes[2].imshow(image_np)
    axes[2].imshow(cam_up, alpha=0.45, aspect="auto", vmin=0.0, vmax=1.0)
    axes[2].set_title(f"Overlay\nPred: {pred_name} ({result['confidence']:.2f})", fontsize=10, pad=8)
    axes[2].axis("off")

    cbar = fig.colorbar(heatmap, cax=cax)
    cbar.ax.set_ylabel("CAM Intensity", rotation=270, labelpad=22)
    cbar.ax.tick_params(labelsize=9)

    fig.suptitle(
        f"Grad-CAM Visualization (Layer Index: {result['layer_index']})",
        fontsize=13,
        y=0.98,
    )

    fig.subplots_adjust(
        left=0.025,
        right=0.975,
        top=0.80,
        bottom=0.06,
        wspace=0.16,
    )

    save_and_show(fig, "grad_cam_heatmap.png")

In [ ]:
# 数据集可视化
plot_dataset_class_distribution(full_dataset, train_dataset, valid_dataset, CLASSES)
show_dataset_samples(full_dataset, CLASSES, samples_per_class=2)
show_augmentation_examples(full_dataset, train_transform, CLASSES, sample_index=0, num_augmented=5)


In [ ]:
# 4. 定义DINO模型
class DINOv2_RemoteSensing_Enhance(nn.Module):
    def __init__(self, num_classes, model_type="facebook/dinov2-base"):
        super().__init__()
        self.backbone = Dinov2Model.from_pretrained(model_type)
        hidden_dim = self.backbone.config.hidden_size
        
        # for i, Dinov2Layer in enumerate(self.backbone.encoder.layer):
        #     if i+1 not in (3, 6, 9, 12):
        #         for name, param in Dinov2Layer.named_parameters():
        #             param.requires_grad = False

        # 多层特征选择权重聚合
        self.layer_weights = nn.Parameter(torch.ones(4))
        self.adapt_weights = nn.Parameter(torch.ones(2))
        # self.layer_weights = nn.Parameter(torch.tensor([0, 0, 0, 1]).float())
        # self.adapt_weights = nn.Parameter(torch.tensor([0, 0, 0, 1]).float())
        
        # Remote Sensing Adapter 低秩自适应，注入领域特定的线性空间映射
        self.adapter = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.GELU(),
            nn.Linear(hidden_dim // 4, hidden_dim)
        )

        self.num_classes = num_classes
        self.memory_momentum = 0.99  # EMA动量
        # 内存库：存储每个类别的原型特征 (num_classes, hidden_dim)
        self.register_buffer('memory_bank', torch.zeros(num_classes, hidden_dim))
        # 记录每个类别已用于更新的样本数（用于首次直接赋值）
        self.register_buffer('class_counts', torch.zeros(num_classes, dtype=torch.long))
        # 可学习的融合权重，平衡原始logits与内存库logits
        self.memory_weight = nn.Parameter(torch.tensor(0.5))
        
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(0.5),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, pixel_values, labels=None):
        outputs = self.backbone(pixel_values=pixel_values, output_hidden_states=True)
        all_hidden_states = outputs.hidden_states
        
        # 提取指定层的 CLS Token
        layers = [all_hidden_states[i-1][:, 0, :] for i in [3, 6, 9, 12]]
        
        # 多尺度特征加权融合
        weighted_feat = 0
        layer_norm_weights = torch.softmax(self.layer_weights, dim=0)
        for i, feat in enumerate(layers):
            weighted_feat += layer_norm_weights[i] * feat
            
        # 通过 Adapter 进行领域特征变换
        adapt_norm_weights = torch.softmax(self.adapt_weights, dim=0)
        adapted_feat = adapt_norm_weights[0] * self.adapter(weighted_feat) \
            + adapt_norm_weights[1] * weighted_feat
    
        orig_logits = self.classifier(adapted_feat)
    
        # 融合两种logits得到最终输出
        memory_bank_current = self.memory_bank.detach().clone()  # 分离并克隆
        memory_logits = adapted_feat @ memory_bank_current.T
        logits = orig_logits + self.memory_weight * memory_logits

        # 训练时：使用真实标签更新内存库（EMA）
        if self.training and labels is not None:
            with torch.no_grad():
                features = adapted_feat.detach()
                unique_labels = labels.unique()
                for c in unique_labels:
                    idx = (labels == c).nonzero(as_tuple=True)[0]
                    if len(idx) == 0:
                        continue
                    class_feats = features[idx]
                    class_mean = class_feats.mean(dim=0)

                    if self.class_counts[c] == 0:
                        self.memory_bank[c] = class_mean
                    else:
                        self.memory_bank[c] = self.memory_momentum * self.memory_bank[c] \
                                              + (1 - self.memory_momentum) * class_mean
                    self.class_counts[c] += len(idx)
        return logits

model = DINOv2_RemoteSensing_Enhance(NUM_CLASSES, "/root/autodl-tmp/models/dinov2-base").to(DEVICE)


In [ ]:
def count_parameters_detailed(model):
    """
    详细统计模型参数：总参数、训练参数，以及各部分的参数分布
    """
    # 初始化统计变量
    total_params = 0
    trainable_params = 0
    
    # 按模块分类统计
    backbone_total = 0
    backbone_trainable = 0
    adapter_total = 0
    memory_total = 0
    classifier_total = 0
    weights_total = 0
    
    # 遍历所有参数
    for name, param in model.named_parameters():
        param_num = param.numel()
        total_params += param_num
        
        # 统计训练参数
        if param.requires_grad:
            trainable_params += param_num
        
        # 按模块分类
        if "backbone" in name:
            backbone_total += param_num
            if param.requires_grad:
                backbone_trainable += param_num
        elif "adapter." in name:
            adapter_total += param_num
        elif "memory_bank" in name:
            memory_total += param_num
        elif "classifier." in name:
            classifier_total += param_num
        elif "layer_weights" in name or "adapt_weights" in name:
            weights_total += param_num
    
    # 单位转换（转成M，百万）
    def to_million(x):
        return x / 1_000_000
    
    # 输出详细结果
    print("="*60)
    print("模型参数统计结果（DINOv2 Remote Sensing Enhance）")
    print("="*60)
    print(f"1. 总参数量: {total_params:,} ({to_million(total_params):.2f}M)")
    print(f"2. 增加的参数量: {total_params - backbone_total:,} ({to_million(total_params - backbone_total):.2f}M)")
    print(f"3. 增加的参数占比: {(total_params - backbone_total)/total_params*100:.4f}%")
    print("-"*60)
    print("各模块参数分布：")
    print(f"- Backbone (DINOv2) 总参数: {backbone_total:,} ({to_million(backbone_total):.2f}M)")
    print(f"- Backbone 可训练参数: {backbone_trainable:,} ({to_million(backbone_trainable):.2f}M)")
    print(f"- Adapter 网络参数: {adapter_total:,} ({to_million(adapter_total):.2f}M)")
    # print(f"- MemoryBank 网络参数: {memory_total:,} ({to_million(adapter_total):.2f}M)")
    print(f"- Classifier 分类头参数: {classifier_total:,} ({to_million(classifier_total):.2f}M)")
    print(f"- 特征权重参数 (layer/adapt): {weights_total:,} ({to_million(weights_total):.6f}M)")
    print("="*60)
    
    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "trainable_ratio": trainable_params/total_params
    }

param_stats = count_parameters_detailed(model)

In [ ]:
# 5. 训练组件
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [ ]:
# 6. 训练循环
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_acc": [],
    "lr": [],
}

# Track one fixed validation sample's DINOv2 token activation map across training.
feature_map_history = []
if len(valid_dataset) > 0:
    fixed_feature_image, fixed_feature_label = valid_dataset[0]
    record_feature_map_snapshot(
        model, fixed_feature_image, fixed_feature_label, DEVICE,
        feature_map_history, epoch=0, layer_index=12
    )

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    current_lr = optimizer.param_groups[0]["lr"]

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images, labels)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)

    train_loss /= len(train_loader.dataset)
    scheduler.step()

    # Validation
    model.eval()
    correct = 0
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data)

    val_loss /= len(valid_loader.dataset)
    val_acc = correct.double() / len(valid_loader.dataset)
    val_acc_value = val_acc.item()

    history["epoch"].append(epoch + 1)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc_value)
    history["lr"].append(current_lr)

    if len(valid_dataset) > 0:
        record_feature_map_snapshot(
            model, fixed_feature_image, fixed_feature_label, DEVICE,
            feature_map_history, epoch=epoch + 1, layer_index=12
        )

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc_value:.4f} | "
        f"LR: {current_lr:.2e}"
    )


In [ ]:
# 7. 保存模型
torch.save(model.state_dict(), "dinov2-cls-RSSCN7.pth")

In [ ]:
# 8. 训练后可视化
plot_training_history(history)

eval_result = evaluate_model_for_visualization(
    model, valid_loader, CLASSES, DEVICE, criterion=criterion, max_store_images=160
)
loss_text = f"{eval_result['loss']:.4f}" if eval_result['loss'] is not None else "N/A"
print(f"Validation Loss: {loss_text} | Validation Acc: {eval_result['acc']:.4f}")

cm = plot_confusion_matrix(eval_result["y_true"], eval_result["y_pred"], CLASSES, normalize=True)
per_class_acc = plot_per_class_accuracy(eval_result["y_true"], eval_result["y_pred"], CLASSES)
report_rows = print_classification_report(eval_result["y_true"], eval_result["y_pred"], CLASSES)

plot_confidence_distribution(eval_result["y_true"], eval_result["y_pred"], eval_result["confidences"])
plot_prediction_examples(eval_result, CLASSES, mode="errors", max_images=12)
plot_prediction_examples(eval_result, CLASSES, mode="correct", max_images=12)

if "feature_map_history" in globals() and len(feature_map_history) > 0:
    plot_feature_map_evolution(feature_map_history, fixed_feature_image, fixed_feature_label, CLASSES, max_snapshots=6)
else:
    print("Feature-map evolution is unavailable because the training loop has not recorded snapshots.")

plot_key_position_feature_maps(model, valid_dataset, CLASSES, DEVICE, sample_index=0, target_class=None)

plot_grad_cam(model, valid_dataset, CLASSES, DEVICE, sample_index=0, target_class=None)
